# Does weather explain the losses in our portfolio?

Each county and season of the book, as the analyst role is served it: the portfolio's loss ratio beside the season's rain and heat and the share of the whole market's losses the USDA puts down to drought and heat. How much of the loss ratio do those measures explain, what does each one add, and what loss ratio does a season like the driest one imply for today's book? On a desktop Jupyter the same code reads large pulls over Arrow Flight.

In [ ]:
%pip install provisa-client
import provisa_client

provisa_client.sign_in()

In [ ]:
import pandas as pd

df = provisa_client.query(
    "SELECT s.county_fips, s.county_name, s.state_abbr, s.year, s.premium, s.indemnity, "
    "s.loss_ratio, s.season_rain_mm, s.hot_days, s.market_dry_share "
    'FROM "agriculture"."view_losses_and_weather" AS s',
    role="agriculture_analyst",
)
measures = ["season_rain_mm", "hot_days", "market_dry_share"]
numeric = ["premium", "indemnity", "loss_ratio", *measures]
df[numeric] = df[numeric].astype(float)
df["year"] = df["year"].astype(int)
seasons = df.dropna(subset=["loss_ratio", *measures])
if seasons.empty:
    raise ValueError("no county season has a loss ratio and all three weather measures")
print(
    len(seasons),
    "county seasons;",
    seasons["county_fips"].nunique(),
    "counties;",
    seasons["year"].min(),
    "to",
    seasons["year"].max(),
)
seasons.head()

In [ ]:
from scipy import stats

rows = []
for m in measures:
    r, p = stats.pearsonr(seasons[m], seasons["loss_ratio"])
    rows.append({"measure": m, "correlation with loss ratio": round(r, 2), "p-value": p})
pd.DataFrame(rows)

In [ ]:
from sklearn.linear_model import LinearRegression

X, y, weight = seasons[measures], seasons["loss_ratio"], seasons["premium"]
model = LinearRegression().fit(X, y, sample_weight=weight)
r2 = model.score(X, y, sample_weight=weight)
rain, heat, dry = model.coef_
print(
    f"The weather explains {r2:.0%} of the variation in loss ratio across county seasons "
    "(weighted by premium)."
)
print(f"Each 100 mm less rain in May to August: {-rain * 100 * 100:+.0f} points of loss ratio.")
print(f"Each 10 days above 32 C: {heat * 10 * 100:+.0f} points of loss ratio.")
print(
    f"Each 10 points more of the market's losses down to drought and heat: "
    f"{dry * 0.10 * 100:+.0f} points of loss ratio."
)

In [ ]:
def portfolio_ratio(frame, ratio):
    return (ratio * frame["premium"]).sum() / frame["premium"].sum()


# The driest season the data holds: the crop year whose county seasons had the least rain.
dry_year = int(seasons.groupby("year")["season_rain_mm"].mean().idxmin())
train, held = seasons[seasons["year"] != dry_year], seasons[seasons["year"] == dry_year]
if train.empty:
    raise ValueError(f"every county season with all three weather measures is from {dry_year}")
without = LinearRegression().fit(
    train[measures], train["loss_ratio"], sample_weight=train["premium"]
)
expected_dry = portfolio_ratio(held, without.predict(held[measures]))
actual_dry = portfolio_ratio(held, held["loss_ratio"])
print(
    f"{dry_year} is the driest season. Fitted without it, the model expects a {dry_year} "
    f"portfolio loss ratio of {expected_dry:.2f}; the book's actual was {actual_dry:.2f}."
)

In [ ]:
# A season like the driest across today's book: each county's own weather that year, at the
# county's latest premium.
latest = df.sort_values("year").groupby("county_fips").tail(1).set_index("county_fips")
book = latest[["premium"]].join(held.set_index("county_fips")[measures], how="inner")
expected = portfolio_ratio(book, model.predict(book[measures]))
print(
    f"Across today's book ({len(book)} counties), a season like {dry_year} has an expected "
    f"portfolio loss ratio of {expected:.2f}."
)

In [ ]:
import matplotlib.pyplot as plt

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))
driest = seasons["year"] == dry_year
left.scatter(
    seasons.loc[~driest, "hot_days"],
    seasons.loc[~driest, "loss_ratio"],
    s=12,
    alpha=0.6,
    label="other seasons",
)
left.scatter(
    seasons.loc[driest, "hot_days"],
    seasons.loc[driest, "loss_ratio"],
    s=16,
    color="crimson",
    label=str(dry_year),
)
left.set_xlabel("days above 32 C, May to August")
left.set_ylabel("loss ratio")
left.legend()
weighted = (
    seasons.assign(
        actual=seasons["loss_ratio"] * seasons["premium"],
        expected=model.predict(seasons[measures]) * seasons["premium"],
    )
    .groupby("year")[["premium", "actual", "expected"]]
    .sum()
)
by_year = weighted[["actual", "expected"]].div(weighted["premium"], axis=0)
by_year.plot(ax=right, marker="o")
right.set_title(f"Portfolio loss ratio by year, R\u00b2 = {r2:.2f}")
right.set_xlabel("crop year")
plt.tight_layout()
plt.show()